# Entrega 2

Un día de posiciones AIS (NOAA) queda en el volume compartido y, desde ahí, en la tabla
bronze. El día lo marca el widget `fecha`. Esta parte guarda el archivo crudo: no corrige
SOG, coordenadas ni tipos de buque. La unica limpieza es una fila por `(MMSI, BaseDateTime)`.

El catálogo `dbr_acad_ingsistemas_mine` y el esquema `default` son los del curso, y no son
compartidos, cada usuario solo ve los que creo. Cada objeto lleva el prefijo `oceanwatch_`. Este notebook es independiente
del de la Parte 1.

Corre en el editor web de Databricks, en el cluster `1114-011114-9wt67574`.

Ejecutar de arriba hacia abajo. Si cambias `fecha`, vuelve a ejecutar la celda de
configuración: el proceso solo ve el valor cuando esa celda se ejecuta.


## Declarar variables para rutas

`fecha` es un widget para que el job programado pueda pasar el día (`YYYY-MM-DD`).


In [0]:
import datetime as dt

dbutils.widgets.text("fecha", "2023-06-08", "Fecha (YYYY-MM-DD)")
fecha = dbutils.widgets.get("fecha").strip()

try:
    fecha_dia = dt.date.fromisoformat(fecha)
except ValueError as exc:
    raise ValueError(f"fecha debe tener la forma YYYY-MM-DD. Llego: {fecha!r}") from exc

# BaseDateTime viene en UTC. El parseo no debe depender de la zona del cluster.
spark.conf.set("spark.sql.session.timeZone", "UTC")

CATALOG = "dbr_acad_ingsistemas_mine"
SCHEMA = "default"
VOLUME = "oceanwatch_data"

VOLUME_NAME = f"`{CATALOG}`.`{SCHEMA}`.`{VOLUME}`"
BRONZE_TABLE = f"`{CATALOG}`.`{SCHEMA}`.`oceanwatch_bronze_ais`"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
RAW_ZIPS = f"{VOLUME_ROOT}/raw_zips"
LANDING = f"{VOLUME_ROOT}/landing"
CHECKPOINTS = f"{VOLUME_ROOT}/checkpoints"
SCHEMAS = f"{VOLUME_ROOT}/_schemas"
REF = f"{VOLUME_ROOT}/ref"
STAGING = f"{RAW_ZIPS}/_staging"
CHECKPOINT = f"{CHECKPOINTS}/bronze_ais"
SCHEMA_LOC = f"{SCHEMAS}/bronze_ais"

CARPETAS = [
    ("raw_zips", RAW_ZIPS),
    ("landing", LANDING),
    ("checkpoints", CHECKPOINTS),
    ("_schemas", SCHEMAS),
    ("ref", REF),
]

print(f"Volume: {VOLUME_ROOT}")
print(f"Bronze: {BRONZE_TABLE}")
print(f"Fecha:  {fecha_dia.isoformat()}")


Volume: /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data
Bronze: `dbr_acad_ingsistemas_mine`.`default`.`oceanwatch_bronze_ais`
Fecha:  2023-06-08


## Configuración, volume y carpetas

| Carpeta | Para qué |
|---|---|
| `raw_zips/` | Zip del día, como lo publica NOAA |
| `landing/` | CSV ya completos. Única ruta que vigila el stream |
| `checkpoints/` | Checkpoint de la capa bronze |
| `_schemas/` | Esquema que persiste Auto Loader |
| `ref/` | Referencias. El World Port Index entra después |

`CREATE VOLUME IF NOT EXISTS` y las carpetas se pueden repetir.

En este cluster (acceso shared) los archivos se leen y se escriben con Python sobre
el mount `/Volumes`. No se usa `dbutils.fs` ni una ruta del disco local del driver:
eso exige el privilegio ANY FILE.


In [0]:
import errno
import os
import shutil


def mover_en_volume(origen: str, destino: str) -> None:
    """Publica origen en destino dentro del mismo volume.

    os.replace en el mismo mount es un rename atomico. Si el FUSE no renombra,
    shutil.move copia y borra.
    """
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    try:
        os.replace(origen, destino)
    except OSError as exc:
        no_rename = {
            errno.EXDEV,
            errno.ENOTSUP,
            getattr(errno, "EOPNOTSUPP", errno.ENOTSUP),
            errno.EPERM,
            errno.ENOSYS,
        }
        if exc.errno not in no_rename:
            raise
        print(f"os.replace no disponible en este mount ({exc}). Se usa shutil.move.")
        shutil.move(origen, destino)


In [0]:
import os

spark.sql(
    f"""
    CREATE VOLUME IF NOT EXISTS {VOLUME_NAME}
    COMMENT 'Archivos de OceanWatch AIS: zips NOAA, CSV de landing, checkpoints, esquemas de Auto Loader y referencias.'
    """
)

for nombre, ruta in CARPETAS:
    os.makedirs(ruta, exist_ok=True)
    print(f"Listo {nombre}: {ruta}")


Listo raw_zips: /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/raw_zips
Listo landing: /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/landing
Listo checkpoints: /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/checkpoints
Listo _schemas: /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/_schemas
Listo ref: /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/ref


## Descarga automatica del día

Se descarga el archivo AIS del la fecha en el widget `fecha`. La se construye asi:

`https://coast.noaa.gov/htdata/CMSP/AISDataHandler/{YYYY}/AIS_{YYYY}_{MM}_{DD}.zip`

1. Si `landing/AIS_YYYY_MM_DD.csv` ya pesa lo que dice el zip, no se descarga ni se extrae.
2. Si `raw_zips/AIS_YYYY_MM_DD.zip` pasa `verify_zip_integrity`, no se vuelve a descargar.
   Si falta o está corrupto, `download_with_retries` lo escribe en `raw_zips/_staging/`
   y solo entonces lo pasa a `raw_zips/`.
3. Si el CSV en `raw_zips/_staging/` ya tiene el tamaño del miembro del zip, no se extrae otra vez. Si no, se extrae ahí.
4. Ese CSV entra a `landing/` con `os.replace`. Si el mount no acepta el rename, se usa `shutil.move`.

Todo el archivo se lee y se escribe sobre el mount `/Volumes`. No hay staging en el disco
local del driver: copiar desde ahí con `dbutils.fs` pide el privilegio ANY FILE.

Para forzar otra descarga hay que borrar el CSV de `landing/` y el zip de `raw_zips/`.
Si el stream ya leyó esa ruta, Auto Loader no la repite. Se podria subir de nuevo con otro nombre pero el MERGE de la sección del stream evita que esa segunda copia duplique filas.


In [0]:
import datetime as dt
import os
import shutil
import time
import zipfile
from pathlib import Path

import requests


def download_with_retries(url: str, dest_path: Path, max_retries: int = 3, delay: int = 5) -> None:
    """Descarga url en dest_path. Reintenta y reemplaza el archivo en cada intento."""
    dest_path.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Descargando {dest_path.name} (intento {attempt}/{max_retries})...")
            with requests.get(url, stream=True, timeout=(30, 120)) as response:
                response.raise_for_status()
                with dest_path.open("wb") as handle:
                    for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                        if chunk:
                            handle.write(chunk)
                    handle.flush()
                    try:
                        os.fsync(handle.fileno())
                    except OSError:
                        pass
            return
        except Exception as exc:
            print(f"Intento {attempt} fallo: {exc}")
            dest_path.unlink(missing_ok=True)
            if attempt == max_retries:
                raise RuntimeError(
                    f"No se pudo descargar {url} despues de {max_retries} intentos."
                ) from exc
            time.sleep(delay)


def verify_zip_integrity(zip_filepath: Path) -> bool:
    """Verifica que el zip sea valido y no este corrupto."""
    print(f"Verificando integridad de {zip_filepath.name}...")
    if not zip_filepath.is_file() or not zipfile.is_zipfile(zip_filepath):
        return False
    with zipfile.ZipFile(zip_filepath, "r") as zf:
        # testzip devuelve None si no hay archivos danados.
        bad_file = zf.testzip()
        return bad_file is None


def _miembros_csv(zf: zipfile.ZipFile) -> list[str]:
    return [
        nombre
        for nombre in zf.namelist()
        if nombre.lower().endswith(".csv")
        and not nombre.endswith("/")
        and "__MACOSX" not in nombre.split("/")
    ]


def tamano_csv_en_zip(zip_path: Path) -> int:
    """Bytes del unico CSV, segun el directorio central del zip (sin descomprimir)."""
    with zipfile.ZipFile(zip_path) as zf:
        miembros = _miembros_csv(zf)
        if len(miembros) != 1:
            raise ValueError(
                f"Se esperaba un CSV dentro de {zip_path.name}; hay {len(miembros)}: {miembros}"
            )
        return zf.getinfo(miembros[0]).file_size


def extract_single_csv(zip_path: Path, dest_csv: Path) -> None:
    """Copia el unico CSV del zip a dest_csv, con el nombre canonico del dia."""
    dest_csv.parent.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        miembros = _miembros_csv(zf)
        if len(miembros) != 1:
            raise ValueError(
                f"Se esperaba un CSV dentro de {zip_path.name}; hay {len(miembros)}: {miembros}"
            )
        with zf.open(miembros[0]) as origen, dest_csv.open("wb") as destino:
            shutil.copyfileobj(origen, destino, length=8 * 1024 * 1024)
            destino.flush()
            try:
                os.fsync(destino.fileno())
            except OSError:
                pass
    if dest_csv.stat().st_size == 0:
        raise ValueError(f"El CSV extraido quedo vacio: {dest_csv.name}")


def ingest_day(day: dt.date) -> str:
    """Descarga, extrae y publica sobre /Volumes."""
    nombre = f"AIS_{day.year}_{day.month:02d}_{day.day:02d}"
    csv_name = f"{nombre}.csv"
    zip_name = f"{nombre}.zip"
    zip_final = Path(RAW_ZIPS) / zip_name
    zip_parcial = Path(STAGING) / f"{zip_name}.partial"
    csv_staging = Path(STAGING) / csv_name
    csv_landing = Path(LANDING) / csv_name

    for carpeta in (RAW_ZIPS, LANDING, STAGING):
        os.makedirs(carpeta, exist_ok=True)

    if zip_final.is_file() and verify_zip_integrity(zip_final):
        print(f"Zip valido en raw_zips ({zip_final.stat().st_size} bytes). No se descarga.")
        zip_parcial.unlink(missing_ok=True)
    else:
        if zip_final.exists():
            print("El zip en raw_zips no paso la integridad. Se vuelve a bajar.")
            zip_final.unlink()
        url = f"https://coast.noaa.gov/htdata/CMSP/AISDataHandler/{day.year}/{zip_name}"
        print(f"Fuente: {url}")
        download_with_retries(url, zip_parcial)
        if not verify_zip_integrity(zip_parcial):
            zip_parcial.unlink(missing_ok=True)
            raise ValueError(
                f"La verificacion de integridad fallo para {zip_name}. El archivo esta corrupto."
            )
        print(f"Integridad verificada para {zip_name} ({zip_parcial.stat().st_size} bytes).")
        mover_en_volume(str(zip_parcial), str(zip_final))
        print(f"Zip guardado en {zip_final}")

    esperado = tamano_csv_en_zip(zip_final)
    if csv_landing.is_file() and csv_landing.stat().st_size == esperado:
        print(f"El CSV de {day.isoformat()} ya esta en landing ({esperado} bytes). No se extrae.")
        print(csv_landing)
        csv_staging.unlink(missing_ok=True)
        return str(csv_landing)

    if csv_staging.is_file() and csv_staging.stat().st_size == esperado:
        print(f"CSV en staging ya completo ({esperado} bytes). No se extrae de nuevo.")
    else:
        extract_single_csv(zip_final, csv_staging)
        print(f"CSV extraido: {csv_staging.name} ({csv_staging.stat().st_size} bytes).")
        if csv_staging.stat().st_size != esperado:
            raise ValueError(f"El CSV extraido mide {csv_staging.stat().st_size} bytes, el zip declara {esperado} bytes.")

    if csv_landing.exists():
        csv_landing.unlink()
    mover_en_volume(str(csv_staging), str(csv_landing))
    print(f"CSV publicado en {csv_landing}")
    return str(csv_landing)


In [0]:
# Idempotente: si el CSV de fecha_dia ya esta en landing, no vuelve a bajar el zip
ingest_day(fecha_dia)

Fuente: https://coast.noaa.gov/htdata/CMSP/AISDataHandler/2023/AIS_2023_06_08.zip
Descargando AIS_2023_06_08.zip.partial (intento 1/3)...
Verificando integridad de AIS_2023_06_08.zip.partial...
Integridad verificada para AIS_2023_06_08.zip (348931376 bytes).
Zip guardado en /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/raw_zips/AIS_2023_06_08.zip
CSV extraido: AIS_2023_06_08.csv (955097423 bytes).
CSV publicado en /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/landing/AIS_2023_06_08.csv


'/Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/landing/AIS_2023_06_08.csv'

## Stream bronze

Auto Loader lee `landing/` con el esquema explícito de la Parte 1. El checkpoint
queda en `checkpoints/bronze_ais` y el esquema de Auto Loader en `_schemas/bronze_ais`.
Se usa `trigger(availableNow=True)` que procesa lo que haya en landing y termina, lo cual sirve para subir un archivo manualmente o desde un job.

La tabla se crea vacía antes del stream y el `MERGE` necesita que exista.
Si una ejecución anterior dejó `oceanwatch_bronze_ais` con otras columnas,
`CREATE TABLE IF NOT EXISTS` no la modifica, entonces hay que borrarla a mano y volver a crearla.

### Duplicados por buque y timestamp

Cada microbatch elimina los duplicados MMSI y BaseDateTime y hace un `MERGE` con una sola
acción: `WHEN NOT MATCHED THEN INSERT`. Esto hace que una fila ya escrita no se actualize ni se borre.

Esta forma maneja un archivo resubido mejor que un Watermark sobre `BaseDateTime`
porque el watermark solo recuerda claves dentro de una ventana de event-time. Pasada la ventana, el mismo archivo se inserta otra vez. Además `foreachBatch` es at-least-once: si el proceso se cae después de escribir y antes de comprometer el offset, Spark repite el batch. Con el `MERGE`, la tabla es la lista de claves y el reintento queda en cero filas nuevas. Esto tambien cubre un CSV con otro nombre, un checkpoint que se reconstruye y los archivos que siguen en `landing/`.

Dentro del batch, si el mismo `(MMSI, BaseDateTime)` trae coordenadas distintas (posiciones incombatibles), bronze conserva la primera que entró. Silver puede revisarlo.

**El stream no se puede correr en dos sesiones a la vez.** El checkpoint es uno solo en un volume compartido y dos queries sobre ese directorio lo corrompen.

Crear tabla bronze

In [0]:
from pyspark.sql import functions as F

# Mismo esquema explicito de la Parte 1
ais_schema = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG double,
    COG double,
    Heading double,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType int,
    Status int,
    Length double,
    Width double,
    Draft double,
    Cargo string,
    TransceiverClass string
"""

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE} (
  MMSI STRING,
  BaseDateTime TIMESTAMP,
  LAT DOUBLE,
  LON DOUBLE,
  SOG DOUBLE,
  COG DOUBLE,
  Heading DOUBLE,
  VesselName STRING,
  IMO STRING,
  CallSign STRING,
  VesselType INT,
  Status INT,
  Length DOUBLE,
  Width DOUBLE,
  Draft DOUBLE,
  Cargo STRING,
  TransceiverClass STRING,
  _source_file STRING COMMENT 'Ruta del CSV en landing (_metadata.file_path).',
  _ingested_at TIMESTAMP COMMENT 'Instante en que el microbatch escribio la fila.'
)
USING DELTA
COMMENT 'Capa bronze de OceanWatch AIS: mensajes crudos e inmutables. Una fila por (MMSI, BaseDateTime), resubir un archivo no inserta otra copia. _source_file y _ingested_at son metadatos de la ingesta.'
""")


DataFrame[]

Correr stream en bronze

In [0]:
from pyspark.sql import functions as F


def leer_landing():
    return (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", SCHEMA_LOC)
        .option("cloudFiles.schemaEvolutionMode", "none")
        .option("cloudFiles.includeExistingFiles", "true")
        .option("cloudFiles.allowOverwrites", "false")
        # false: listar landing completo. El default incremental se salta un archivo
        # nuevo que no queda delante del ultimo ya ingerido (nombre y mtime).
        .option("cloudFiles.useIncrementalListing", "false")
        .option("header", "true")
        .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
        .schema(ais_schema)
        .load(LANDING)
        .select(
            "MMSI", "BaseDateTime", "LAT", "LON", "SOG", "COG", "Heading", "VesselName", "IMO",
            "CallSign", "VesselType", "Status", "Length", "Width", "Draft", "Cargo", "TransceiverClass",
            F.col("_metadata.file_path").alias("_source_file"),
            F.current_timestamp().alias("_ingested_at"),
        )
    )


def merge_into_bronze(batch_df, batch_id: int) -> None:
    """Inserta claves nuevas. Una fila ya presente no se actualiza.

    Solo cierra sobre BRONZE_TABLE para que foreachBatch pueda serializarla.
    """
    sesion = batch_df.sparkSession
    sin_duplicados = batch_df.dropDuplicates(["MMSI", "BaseDateTime"])
    vista = f"oceanwatch_bronze_batch_{int(batch_id)}"
    sin_duplicados.createOrReplaceTempView(vista)
    sesion.sql(
        f"""
        MERGE INTO {BRONZE_TABLE} AS destino
        USING {vista} AS origen
        ON destino.MMSI = origen.MMSI
       AND destino.BaseDateTime = origen.BaseDateTime
        WHEN NOT MATCHED THEN INSERT *
        """
    )
    sesion.catalog.dropTempView(vista)


def detener_consulta_previa() -> None:
    for consulta in list(spark.streams.active):
        if consulta.name == "oceanwatch_bronze_ais":
            print(
                "Consulta bronze activa en esta sesion. "
                "Se detiene y se retoma desde el checkpoint."
            )
            consulta.stop()


def filas_bronze() -> int:
    fila = spark.sql(f"SELECT COUNT(*) AS n FROM {BRONZE_TABLE}").collect()[0]
    return int(fila["n"])


def filas_leidas(query) -> int:
    eventos = list(query.recentProgress or [])
    if not eventos and getattr(query, "lastProgress", None):
        eventos = [query.lastProgress]
    total = 0
    for evento in eventos:
        if isinstance(evento, dict):
            n = int(evento.get("numInputRows") or 0)
            batch_id = evento.get("batchId")
        else:
            n = int(getattr(evento, "numInputRows", 0) or 0)
            batch_id = getattr(evento, "batchId", None)
        total += n
        print(f"batch {batch_id}: numInputRows={n}")
    return total


def run_bronze_stream():
    print(f"Checkpoint: {CHECKPOINT}")
    print("No ejecutes este stream en paralelo con otra sesion.")
    detener_consulta_previa()
    consulta = (
        leer_landing()
        .writeStream.foreachBatch(merge_into_bronze)
        .queryName("oceanwatch_bronze_ais")
        .option("checkpointLocation", CHECKPOINT)
        .trigger(availableNow=True)
        .start()
    )
    consulta.awaitTermination()
    return consulta


antes = filas_bronze()
consulta = run_bronze_stream()
leidas = filas_leidas(consulta)
despues = filas_bronze()
print(f"Ejecucion: leidas={leidas} antes={antes} despues={despues}")
if leidas == 0 and antes == 0:
    print("No entro ningun archivo. Revisa que landing tenga el CSV del dia.")


Checkpoint: /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/checkpoints/bronze_ais
No ejecutes este stream en paralelo con otra sesion.
batch 0: numInputRows=0
Ejecucion: leidas=0 antes=0 despues=8920181
No entro ningun archivo. Revisa que landing tenga el CSV del dia.


## Verificación

Conteo de filas por fecha en UTC, una muestra con las columnas
de la ingesta, y cuántas claves `(MMSI, BaseDateTime)` quedaron repetidas.

In [0]:
from pyspark.sql import functions as F

total = filas_bronze()
print(f"Filas en bronze: {total}")

por_fecha = (
    spark.table(BRONZE_TABLE)
    .groupBy(F.to_date(F.col("BaseDateTime")).alias("fecha"))
    .agg(F.count("*").alias("filas"))
    .orderBy("fecha")
)
display(por_fecha)

display(
    spark.table(BRONZE_TABLE)
    .select("MMSI", "BaseDateTime", "LAT", "LON", "_source_file", "_ingested_at")
    .limit(5)
)

repetidas = spark.sql(
    f"""
    SELECT COUNT(*) AS claves_repetidas
    FROM (
      SELECT MMSI, BaseDateTime
      FROM {BRONZE_TABLE}
      WHERE MMSI IS NOT NULL AND BaseDateTime IS NOT NULL
      GROUP BY MMSI, BaseDateTime
      HAVING COUNT(*) > 1
    ) repetidas
    """
)
display(repetidas)
n_repetidas = int(repetidas.collect()[0]["claves_repetidas"])
print(f"Claves (MMSI, BaseDateTime) repetidas: {n_repetidas}")

if total == 0:
    raise AssertionError(
        "Bronze tiene 0 filas. Revisa la descarga y que el stream haya leido landing."
    )
if n_repetidas != 0:
    raise AssertionError(
        f"Hay {n_repetidas} claves repetidas. El MERGE debio dejar una fila por "
        "(MMSI, BaseDateTime)."
    )


Filas en bronze: 8920181


fecha,filas
2023-06-08,8920181


MMSI,BaseDateTime,LAT,LON,_source_file,_ingested_at
1,2023-06-08T00:10:05Z,49.1253,-123.08509,/Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/landing/AIS_2023_06_08.csv,2026-11-15T18:58:39.911Z
1,2023-06-08T01:49:05Z,49.12531,-123.0851,/Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/landing/AIS_2023_06_08.csv,2026-11-15T18:58:39.911Z
1,2023-06-08T02:19:05Z,49.12531,-123.0851,/Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/landing/AIS_2023_06_08.csv,2026-11-15T18:58:39.911Z
1,2023-06-08T02:34:04Z,49.12531,-123.0851,/Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/landing/AIS_2023_06_08.csv,2026-11-15T18:58:39.911Z
1,2023-06-08T02:49:04Z,49.12531,-123.0851,/Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/landing/AIS_2023_06_08.csv,2026-11-15T18:58:39.911Z


claves_repetidas
0


Claves (MMSI, BaseDateTime) repetidas: 0


## Tolerancia a fallos

Si la celda del stream se interrumpe, volver a ejecutarla retoma desde el checkpoint.
`run_bronze_stream` detiene primero una consulta con el mismo nombre que haya quedado
activa en esta sesión, y Auto Loader no vuelve a leer archivos ya procesados. Un batch
que alcanzó a hacer `MERGE` y luego se repite no duplica filas porque la clave ya está en la tabla.


La celda siguiente es la primera prueba: al ejecutar el stream otra vez con el CSV del dia ya procesado, las filas leidas tiene que ser 0 y el conteo de la tabla igual porque el checkpoint ya marcó el archivo.

In [0]:
antes = filas_bronze()
consulta = run_bronze_stream()
leidas = filas_leidas(consulta)
despues = filas_bronze()
print(f"Segunda ejecución: leidas={leidas} antes={antes} despues={despues}")
if leidas != 0 or despues != antes:
    raise AssertionError(
        "Se esperaba la segunda ejecución con 0 filas nuevas "
        f"(leidas={leidas}, antes={antes}, despues={despues}). "
        "Si el stream todavía no se había ejecutado, ejecuta antes la celda que lo arranca."
    )
print("Checkpoint: la segunda ejecucion no inserto filas.")


Checkpoint: /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/checkpoints/bronze_ais
No ejecutes este stream en paralelo con otra sesion.
batch 1: numInputRows=0
Segunda ejecución: leidas=0 antes=8920181 despues=8920181
Checkpoint: la segunda ejecucion no inserto filas.


La segunda prueba es un archivo nuevo con claves que ya están en bronze. Copiar el CSV
entero no sirve: `shutil.copyfile` sobre el volume deja un objeto cuyo mtime no queda
delante del último archivo ingerido, el listado incremental no lo abre y el batch sale
con `numInputRows = 0`. Aquí se escriben las primeras 1000 filas de datos en un CSV nuevo
(otro nombre, mtime de ahora). El stream tiene que leerlas, y el `MERGE` no inserta ninguna.
Al final se borra ese archivo.

In [0]:
import os

origen = (
    f"{LANDING}/AIS_{fecha_dia.year}_{fecha_dia.month:02d}_{fecha_dia.day:02d}.csv"
)
if not os.path.isfile(origen):
    raise FileNotFoundError(
        f"No esta el CSV del dia en landing ({origen}). Ejecuta antes la descarga."
    )

# Crear archivo de prueba
marca = dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%S")
copia = f"{LANDING}/redrop-{marca}.csv"
n_muestra = 1000
escritas = 0
with open(origen, "r", encoding="utf-8", newline="") as entrada, open(
    copia, "w", encoding="utf-8", newline=""
) as salida:
    encabezado = entrada.readline()
    if not encabezado:
        raise ValueError(f"El CSV de landing esta vacio: {origen}")
    salida.write(encabezado)
    for linea in entrada:
        if escritas >= n_muestra:
            break
        salida.write(linea)
        escritas += 1
    salida.flush()
    os.fsync(salida.fileno())
os.utime(copia, None)

print(f"Muestra de reenvio: {copia} ({escritas} filas)")
try:
    antes = filas_bronze()
    consulta = run_bronze_stream()
    leidas = filas_leidas(consulta)
    despues = filas_bronze()
    print(f"Reenvio: leidas={leidas} antes={antes} despues={despues}")
    if escritas == 0:
        raise AssertionError(f"No se pudieron leer filas de {origen}.")
    if despues != antes:
        raise AssertionError(
            f"El reenvio inserto filas (antes={antes}, despues={despues}). "
            "El MERGE por (MMSI, BaseDateTime) debio dejar la tabla igual."
        )
    if leidas != escritas:
        raise AssertionError(
            f"El stream debio leer las {escritas} filas de la muestra pero leyo {leidas}."
        )
    print("Reenvio sin filas nuevas. La clave ya estaba en bronze.")
finally:
    if os.path.isfile(copia):
        os.remove(copia)
    print(f"Muestra eliminada: {copia}")


Muestra de reenvio: /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/landing/redrop-20261115T190318.csv (1000 filas)
Checkpoint: /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/checkpoints/bronze_ais
No ejecutes este stream en paralelo con otra sesion.
batch 1: numInputRows=0
Reenvio: leidas=0 antes=8920181 despues=8920181
Muestra eliminada: /Volumes/dbr_acad_ingsistemas_mine/default/oceanwatch_data/landing/redrop-20261115T190318.csv


---------------------------------------------------------------------------
AssertionError                            Traceback (most recent call last)
File <command-8864818040331345>, line 47
     42         raise AssertionError(
     43             f"El reenvio inserto filas (antes={antes}, despues={despues}). "
     44             "El MERGE por (MMSI, BaseDateTime) debio dejar la tabla igual."
     45         )
     46     if leidas != escritas:
---> 47         raise AssertionError(
     48             f"El stream debio leer las {escritas} filas de la muestra pero leyo {leidas}."
     49         )
     50     print("Reenvio sin filas nuevas. La clave ya estaba en bronze.")
     51 finally:

AssertionError: El stream debio leer las 1000 filas de la muestra pero leyo 0.